<a href="https://colab.research.google.com/github/NilsChabloz/2026_Intro_Python/blob/main/live/part-I/1.7-oop-for-environmental-systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> Notebook-friendly copy of `part-I/1.7-oop-for-environmental-systems.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [11]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle start in an empty working directory.
# This is a no-op in an environment that is already set up.
from pathlib import Path

Path("_files").mkdir(exist_ok=True)   # the folder this notebook writes into

# 1.7) Object-Oriented and Defensive Programming

Some data naturally travels with the operations that act on it: a weather station has a name, an elevation, and a growing list of readings, and the things you do with it — add a reading, compute its mean — belong to that station. A class bundles such state and behaviour into one object. This subchapter builds a `WeatherStation`, composes stations into a network, contrasts a lightweight `dataclass` record, and is careful about the opposite lesson too: when a plain function is the better tool. It then turns from writing code to trusting it: `assert` for internal invariants versus exceptions for bad input, validating physical preconditions, logging instead of printing, and testing with pytest — the concrete habits for checking code you did not write yourself, including code an AI assistant generated for you.

**🎯 Learning objectives**

- Define a class with `__init__` and `self`, create instances, and validate arguments in the constructor.
- Distinguish instance attributes (unique per object) from class attributes (shared).
- Write type-hinted methods that read state and methods that mutate it.
- Recognise dunder methods, and write a useful `__repr__`.
- Use a dataclass for a lightweight record, and composition to build larger objects from smaller ones.
- Extend a class with inheritance, calling `super().__init__` to reuse the parent's constructor.
- Judge when a class earns its keep and when a function is clearer.
- Use assert to check invariants that should hold if the code is correct.
- Signal and handle bad input with exceptions: raise, custom exception types, and try/except/else/finally.
- Validate physical preconditions explicitly.
- Prefer logging to print for messages that carry a severity level.
- Write and run a pytest suite with plain tests, a fixture, and parametrisation.

## 1.7.1 Object-Oriented Programming

### Classes and Instances

Every value used so far — a string, a list, a numpy array, a DataFrame — is an *object*: data bundled together with the operations that work on it, reached with dot notation (`text.upper()`, `data.mean()`). Until now those objects were always ones Python or a library already defined. A **class** is the blueprint that defines a new kind of object: what data it carries (its *attributes*) and what it can do (its *methods*). An **instance** is one specific object built from that blueprint — `jungfraujoch` and `basel` below are two different instances of the same `WeatherStation` class, each with its own name, elevation, and list of readings.

`__init__` is the *constructor*: the method Python calls automatically each time a new instance is created, and its job is to set up that instance's starting state. Inside any method, `self` refers to the particular instance the method was called on — it is how `jungfraujoch.add_reading(0.4)` updates only `jungfraujoch`'s readings, and `basel.add_reading(18.0)` only `basel`'s.

#### A class to represent a weather station

The smallest useful class stores a single attribute:

In [12]:
class WeatherStation:
    def __init__(self, name: str):
        self.name = name

An instance is created by calling the class like a function, with the arguments `__init__` expects after `self`:

In [13]:
basel = WeatherStation("Basel")
print(basel, basel.name)

<__main__.WeatherStation object at 0x7fcb698f6510> Basel


Printing the object itself shows only its class and a memory address; `basel.name` is the data it holds. A station needs more than a name. The version below adds an elevation and a list for readings, and checks the elevation as the object is built. Every station passes through `__init__`, so a check there means no station with an impossible elevation can exist.

In [14]:
class WeatherStation:
    def __init__(self, name: str, elevation_m: float):
        # Dead Sea shore to the summit of Everest: anything outside is a data-entry error
        if elevation_m < -430 or elevation_m > 8849:
            raise ValueError(f"implausible elevation {elevation_m} m")
        self.name = name                          # instance attributes: per-object
        self.elevation_m = elevation_m
        self.readings_celsius: list[float] = []   # each station owns its own list


basel = WeatherStation("Basel", 316)
print(basel.name, basel.elevation_m, basel.readings_celsius)

Basel 316 []


In [15]:
# 35 710 m is a typo for 3571 m; the constructor refuses it
#WeatherStation("Jungfraujoch", 35710)

Next, the methods: functions defined inside the class, which act on one instance through `self`. Some read the instance's state, some change it, and one below also uses a *class* attribute, a value defined once on the class and shared by every station.

In [16]:
# the built-in sum() adds up the elements of an iterable
print(sum([1.0, 2.0, 3.0]))

6.0


In [17]:
class WeatherStation:
    # class attribute: one value shared by every station (a physical constant)
    lapse_rate_celsius_per_km = -6.5

    def __init__(self, name: str, elevation_m: float):
        # Dead Sea shore to the summit of Everest: anything outside is a data-entry error
        if elevation_m < -430 or elevation_m > 8849:
            raise ValueError(f"implausible elevation {elevation_m} m")
        self.name = name                          # instance attributes: per-object
        self.elevation_m = elevation_m
        self.readings_celsius: list[float] = []   # each station owns its own list

    def add_reading(self, temp_celsius: float) -> None:
        # a method that MUTATES state
        self.readings_celsius.append(temp_celsius)

    def mean_temperature(self) -> float:
        # a method that READS state
        if not self.readings_celsius:
            raise ValueError(f"{self.name} has no readings")
        return sum(self.readings_celsius) / len(self.readings_celsius)

    def sea_level_temperature(self, temp_celsius: float) -> float:
        # uses the shared class attribute together with instance state
        return temp_celsius - self.lapse_rate_celsius_per_km * (self.elevation_m / 1000)

In [18]:
jungfraujoch = WeatherStation("Jungfraujoch", 3571)
for t in [-2.3, -1.1, 0.4]:
    jungfraujoch.add_reading(t)

print(jungfraujoch.readings_celsius)
print("JFJ mean:", round(jungfraujoch.mean_temperature(), 2), "°C")
print("JFJ at sea level:", round(jungfraujoch.sea_level_temperature(0.0), 1), "°C")

[-2.3, -1.1, 0.4]
JFJ mean: -1.0 °C
JFJ at sea level: 23.2 °C


#### Magic (dunder) methods

Methods whose names begin and end with a double underscore — *dunder* methods, also called magic or special methods — are not usually called by name. Python calls them itself, in response to built-in operations, and defining one customises how instances of the class behave. `__init__` is the first: Python calls it whenever a new instance is created. `__repr__` is another: Python calls it whenever it needs a printable representation of an object, as `print` does, and without it you get the class name and memory address seen above. Inside an f-string, `!r` (used in the next cell) inserts the `repr` of a value instead of its plain text, so a string keeps its quotes: `'Basel'` rather than `Basel`. The full list is in the [Python data model](https://docs.python.org/3/reference/datamodel.html#special-method-names).

In [19]:
class WeatherStation:
    # class attribute: one value shared by every station (a physical constant)
    lapse_rate_celsius_per_km = -6.5

    def __init__(self, name: str, elevation_m: float):
        # Dead Sea shore to the summit of Everest: anything outside is a data-entry error
        if elevation_m < -430 or elevation_m > 8849:
            raise ValueError(f"implausible elevation {elevation_m} m")
        self.name = name                          # instance attributes: per-object
        self.elevation_m = elevation_m
        self.readings_celsius: list[float] = []   # each station owns its own list

    def add_reading(self, temp_celsius: float) -> None:
        # a method that MUTATES state
        self.readings_celsius.append(temp_celsius)

    def mean_temperature(self) -> float:
        # a method that READS state
        if not self.readings_celsius:
            raise ValueError(f"{self.name} has no readings")
        return sum(self.readings_celsius) / len(self.readings_celsius)

    def sea_level_temperature(self, temp_celsius: float) -> float:
        # uses the shared class attribute together with instance state
        return temp_celsius - self.lapse_rate_celsius_per_km * (self.elevation_m / 1000)

    def __repr__(self) -> str:
        return f"WeatherStation({self.name!r}, {self.elevation_m} m, n={len(self.readings_celsius)})"

In [20]:
jungfraujoch = WeatherStation("Jungfraujoch", 3571)
basel = WeatherStation("Basel", 316)

for t in [-2.3, -1.1, 0.4]:
    jungfraujoch.add_reading(t)
for t in [18.0, 19.2, 17.5]:
    basel.add_reading(t)

print(jungfraujoch)                                  # __repr__ in action
print(basel)
print("JFJ mean:", round(jungfraujoch.mean_temperature(), 2), "°C")
print("JFJ at sea level:", round(jungfraujoch.sea_level_temperature(0.0), 1), "°C")

WeatherStation('Jungfraujoch', 3571 m, n=3)
WeatherStation('Basel', 316 m, n=3)
JFJ mean: -1.0 °C
JFJ at sea level: 23.2 °C


### Instance versus Class Attributes

An *instance* attribute belongs to one object (each station's own `readings_celsius`). A *class* attribute is shared by every instance (the single `lapse_rate_celsius_per_km`). Shared **constants** are a good use of class attributes; shared **mutable** state can lead to bugs.

In [21]:
# the lapse rate is one shared value, reachable via the class or any instance
print(WeatherStation.lapse_rate_celsius_per_km,
      jungfraujoch.lapse_rate_celsius_per_km,
      basel.lapse_rate_celsius_per_km)

# the readings are independent per instance
print("JFJ readings:", jungfraujoch.readings_celsius)
print("Basel readings:", basel.readings_celsius)

-6.5 -6.5 -6.5
JFJ readings: [-2.3, -1.1, 0.4]
Basel readings: [18.0, 19.2, 17.5]


### dataclasses: Records with Less Ceremony

When an object is mostly a bundle of fields, `@dataclass` generates `__init__`, `__repr__`, and `__eq__` for you from type-annotated attributes.

In [22]:
from dataclasses import dataclass

@dataclass
class Reading:
    timestamp: str
    temp_celsius: float

r = Reading("2024-06-01", 18.2)
print(r)                                                  # auto __repr__
print(r.temp_celsius)
print(Reading("2024-06-01", 18.2) == Reading("2024-06-01", 18.2))   # auto __eq__

Reading(timestamp='2024-06-01', temp_celsius=18.2)
18.2
True


<details>
<summary><b>🔍 Going deeper: composition, building larger objects from smaller ones</b></summary>

Composition is a *has-a* relationship: a `StationNetwork` *has* stations. The container delegates work to the objects it holds rather than re-implementing it.

```python
class StationNetwork:
    def __init__(self, name: str):
        self.name = name
        self.stations: dict[str, WeatherStation] = {}

    def add_station(self, station: WeatherStation) -> None:
        self.stations[station.name] = station

    def coldest_station(self) -> WeatherStation:
        # delegate to each station's own mean_temperature method
        coldest = None
        for station in self.stations.values():   # .values() gives the dict's values: the station objects
            if coldest is None or station.mean_temperature() < coldest.mean_temperature():
                coldest = station
        return coldest

    def __repr__(self) -> str:
        return f"StationNetwork({self.name!r}, {len(self.stations)} stations)"

network = StationNetwork("Switzerland")
network.add_station(jungfraujoch)
network.add_station(basel)
print(network)
print("coldest:", network.coldest_station().name)
```

</details>

**🧠 Computational-thinking fundamental: reach for a class only when there is state to bundle**

A class earns its keep when data and the operations on it belong together and the object carries state between calls — a station accumulating readings, a model remembering what it learned. When there is no state to keep, an object is just ceremony around a function. Converting celsius to kelvin needs no class: there is nothing to remember between calls, so a plain function does the job with less code. Wrapping a stateless calculation in a class anyway adds a constructor to write, an object to instantiate, and attributes to track for data that never changes — more code to read and maintain for no behaviour gained.

### When Not to Use a Class

A stateless transformation needs no object. Wrapping a one-line conversion in a class adds boilerplate and hides a simple function behind a constructor.

In [23]:
# a pure function is the right tool here: input in, output out, no state
def celsius_to_kelvin(temp_celsius: float) -> float:
    return temp_celsius + 273.15

print(celsius_to_kelvin(18.2))
# a class with no attributes and a single method would only add ceremony

291.34999999999997


### Inheritance: Extending a Class

A subclass inherits a parent's attributes and methods — an *is-a* relationship, as opposed to composition's *has-a*. Calling `super().__init__` reuses the parent's constructor, so the subclass only has to set up what is new.

In [24]:
class RiverGauge(WeatherStation):
    def __init__(self, name: str, elevation_m: float):
        super().__init__(name, elevation_m)         # reuse the parent's constructor
        self.discharge_m3s: list[float] = []        # extra state specific to a gauge

    def add_discharge(self, q_m3s: float) -> None:
        self.discharge_m3s.append(q_m3s)

rhone = RiverGauge("Rhone at Porte du Scex", 375)
rhone.add_reading(12.4)          # inherited from WeatherStation
rhone.add_discharge(180.0)
print(rhone.name, rhone.readings_celsius, rhone.discharge_m3s)
print("mean temperature:", round(rhone.mean_temperature(), 2))   # inherited method

Rhone at Porte du Scex [12.4] [180.0]
mean temperature: 12.4


Prefer composition to deep inheritance hierarchies; inherit only for a genuine *is-a* relationship.

<details>
<summary><b>🔍 Going deeper: property validation</b></summary>

A `@property` lets an attribute run validation on assignment while still being accessed like plain data.

```python
class Station:
    def __init__(self, elevation_m):
        self.elevation_m = elevation_m       # goes through the setter

    @property
    def elevation_m(self):
        return self._elevation_m

    @elevation_m.setter
    def elevation_m(self, value):
        if value < -500:
            raise ValueError("elevation below -500 m is implausible")
        self._elevation_m = value
```

</details>

<details>
<summary><b>🔍 Going deeper: abstract base classes</b></summary>

An abstract base class defines an interface that subclasses must implement, enforced at instantiation.

```python
from abc import ABC, abstractmethod

class Sensor(ABC):
    @abstractmethod
    def read(self) -> float:
        ...
# a subclass that does not implement read() cannot be instantiated
```

ABCs document the contract a family of objects must satisfy.

</details>

<details>
<summary><b>🔍 Going deeper: the scikit-learn estimator-as-object pattern</b></summary>

scikit-learn models are objects that learn state in `fit` (stored on attributes with a trailing underscore) and use it in `predict` — the bridge to the machine-learning subchapter.

```python
class MeanRegressor:
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)     # learned state
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]
```

Every estimator you will meet follows this fit/predict object shape.

</details>

## 1.7.2 Defensive Programming

### From Designing Objects to Trusting Code

The first half of this subchapter was about building your own objects. The second is a different skill: reading and stress-testing code you did not write yourself — including code generated by an AI assistant, which can run and look plausible while still being wrong on exactly the input you didn't try. `assert`, exceptions, explicit validation, logging, and tests are the concrete tools for that: ways to make a piece of code state its own assumptions and get caught the moment those assumptions break, rather than trusting that code runs correctly just because it runs.

### assert for Invariants

An `assert` documents and checks a condition that should always be true *if the code is correct*. It is a statement about the program's own logic, not about external input — and it is removed when Python runs with the `-O` flag.

In [25]:
def normalise(weights: list[float]) -> list[float]:
    total = sum(weights)
    result = []
    for w in weights:
        result.append(w / total)
    # invariant: normalised weights sum to 1 (a check on our own arithmetic)
    assert abs(sum(result) - 1.0) < 1e-9, "normalisation failed"
    return result

print(normalise([1.0, 3.0]))

[0.25, 0.75]


### Exceptions: raise, Custom Types, and try/except/else/finally

An exception signals a runtime problem. `raise` triggers one; a custom exception subclass names a specific failure; `try/except/else/finally` handles it — `else` runs when no exception occurred, `finally` always runs.

In [26]:
class PhysicalRangeError(ValueError):
    # a named failure mode, more specific than a bare ValueError
    pass

In [27]:
def to_kelvin(temp_celsius: float) -> float:
    if temp_celsius < -273.15:
        raise PhysicalRangeError(f"{temp_celsius} °C is below absolute zero")
    return temp_celsius + 273.15

In [28]:
for value in [25.0, -300.0]:
    try:
        kelvin = to_kelvin(value)
    except PhysicalRangeError as err:
        print("rejected:", err)
    else:
        print("ok:", round(kelvin, 2), "K")   # runs only when no exception was raised
    finally:
        print("checked", value)               # always runs

ok: 298.15 K
checked 25.0
rejected: -300.0 °C is below absolute zero
checked -300.0


### Validating Physical Preconditions

External input — a file, a user value, a network response — must be validated with exceptions, not asserts, because it can be wrong even when the code is correct.

In [29]:
def validate_observation(temp_celsius: float, discharge_m3s: float) -> None:
    if temp_celsius < -273.15:
        raise PhysicalRangeError("temperature below absolute zero")
    if discharge_m3s < 0.0:
        raise ValueError("discharge cannot be negative")

validate_observation(18.0, 45.0)          # valid input passes silently
try:
    validate_observation(18.0, -5.0)
except ValueError as err:
    print("caught:", err)

caught: discharge cannot be negative


### Logging over print

`print` writes unconditionally to stdout. `logging` attaches a severity level to each message, so the same code can be verbose while debugging and quiet in production, and can route messages to files or services without edits.

In [30]:
import logging
import sys

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
logger = logging.getLogger("mlees")

logger.debug("suppressed: below the INFO threshold")
logger.info("loaded 45 readings")
logger.warning("3 temperature values missing")

INFO: loaded 45 readings


### Testing with pytest

pytest discovers functions named `test_*`, runs them, and reports failures with readable assertion output. A *fixture* supplies reusable setup; *parametrisation* runs one test over many cases. Here we write a small module and its test file to disk, then run the suite with `subprocess.run`, which starts pytest as a separate process (`sys.executable -m pytest` uses the same Python as the notebook) and, with `capture_output=True, text=True`, collects what it prints as strings.

In [31]:
from pathlib import Path

In [32]:
Path("_files").mkdir(exist_ok=True)

In [33]:
module_src = """
def to_kelvin(temp_celsius):
    if temp_celsius < -273.15:
        raise ValueError("below absolute zero")
    return temp_celsius + 273.15
"""
Path("_files/thermo.py").write_text(module_src, encoding="utf-8")

142

In [34]:
test_src = """
import pytest
from thermo import to_kelvin

def test_freezing_point():
    assert to_kelvin(0.0) == 273.15

@pytest.fixture
def boiling_celsius():
    return 100.0

def test_with_fixture(boiling_celsius):
    assert to_kelvin(boiling_celsius) == 373.15

@pytest.mark.parametrize("celsius, kelvin", [(0.0, 273.15), (-273.15, 0.0), (25.0, 298.15)])
def test_conversions(celsius, kelvin):
    assert to_kelvin(celsius) == pytest.approx(kelvin)

def test_below_absolute_zero_raises():
    with pytest.raises(ValueError):
        to_kelvin(-300.0)
"""
Path("_files/test_thermo.py").write_text(test_src, encoding="utf-8")
print("wrote thermo.py and test_thermo.py")

wrote thermo.py and test_thermo.py


In [35]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_thermo.py", "-q", "--color=no"],
    capture_output=True, text=True, cwd="_files",
)
# stderr is printed too, so that a failure to start pytest is visible instead of silent
print((result.stdout + result.stderr).strip())

......                                                                   [100%]
6 passed in 0.02s


## *When generated code lies: the shared class-level list*

Asked for a station class, an assistant declares the readings list at the class level. That single list is then shared by every instance — the object-oriented version of the mutable-default-argument bug.

In [36]:
class WeatherStationBuggy:
    readings_celsius = []        # class attribute: ONE list shared by all instances

    def __init__(self, name):
        self.name = name

    def add_reading(self, temp_celsius):
        self.readings_celsius.append(temp_celsius)

a = WeatherStationBuggy("A")
b = WeatherStationBuggy("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.readings_celsius)   # expected [10.0]
print("B readings:", b.readings_celsius)   # expected [20.0]

A readings: [10.0, 20.0]
B readings: [10.0, 20.0]


**⚠️ Diagnosis: mutable state declared on the class is shared**

`readings_celsius = []` sits on the class, so there is exactly one list and every instance appends to it; station A and station B end up with each other's data, silently. Mutable per-instance state must be created inside `__init__` with `self.readings_celsius = []`, so each object gets its own list. Class attributes are for values that are genuinely shared and, ideally, immutable.

In [37]:
class WeatherStationFixed:
    def __init__(self, name):
        self.name = name
        self.readings_celsius = []      # per-instance list, created on construction

    def add_reading(self, temp_celsius):
        self.readings_celsius.append(temp_celsius)

a = WeatherStationFixed("A")
b = WeatherStationFixed("B")
a.add_reading(10.0)
b.add_reading(20.0)
print("A readings:", a.readings_celsius, "| B readings:", b.readings_celsius)

A readings: [10.0] | B readings: [20.0]


**📌 Takeaways**

- A class bundles state (attributes) and behaviour (methods); `__init__` stores data on `self` at construction.
- Validate arguments in `__init__`, so that an object with impossible values can never be built.
- Instance attributes are per-object; class attributes are shared — good for constants, dangerous for mutable state.
- Dunder methods such as `__init__` and `__repr__` are called by Python itself; write a `__repr__` for readable objects.
- Type-hint methods, and separate methods that read state from those that mutate it.
- Use a `dataclass` for plain records, and composition to assemble larger objects from smaller ones.
- A subclass inherits a parent's attributes and methods; call `super().__init__` to reuse its constructor, then add what's new.
- A stateless transformation should be a function, not a class.
- Never put mutable state (`readings = []`) on the class body: it is shared across all instances. Initialise it in `__init__`.
- `assert` checks internal invariants and is stripped by `python -O`; never use it to validate external input.
- Raise exceptions (a custom subclass when it clarifies intent) for bad input; handle with try/except/else/finally.
- Validate physical preconditions explicitly, and fail early and loudly.
- Prefer `logging` to `print`: messages gain a severity level and can be filtered or redirected.
- Test with pytest — plain tests, fixtures for setup, parametrisation for many cases.

## Summary

| Concept | Rule to remember |
|---|---|
| Classes | Bundle state (attributes) with behaviour (methods); `__init__` stores data on `self`. |
| Attributes | Instance attributes are per-object, class attributes are shared — never put mutable state on the class body. |
| Records | Use a `dataclass` for plain records, and composition to assemble larger objects from smaller ones. |
| Inheritance | A subclass extends a parent; `super().__init__` reuses its constructor. |
| When not to | A stateless transformation should be a function, not a class. |
| `assert` | For internal invariants only — `python -O` strips it, so never validate external input with it. |
| Exceptions | `raise` for bad input, `try`/`except`/`else`/`finally` to handle it; fail early and loudly. |
| Logging | Prefer `logging` to `print`: messages gain a severity level and can be filtered or redirected. |
| Testing | pytest — plain tests, fixtures for setup, parametrisation for many cases. |

## Resources

- [Object-Oriented Programming (OOP) in Python](https://realpython.com/python3-object-oriented-programming/) — classes, instances, attributes, methods, and inheritance, with worked examples.
- [Python Classes: The Power of Object-Oriented Programming](https://realpython.com/python-classes/) — instance vs class attributes, dataclasses, abstract base classes, and explicit guidance on when *not* to use a class.
- [pytest documentation](https://docs.pytest.org/en/stable/) — writing tests, fixtures, parametrisation, and assertions.